# 03 -- SAE Feature Analysis

Sparse autoencoder differential activation analysis. For each SAE feature, test
whether its activation differs between conflict (S1-lure) and no-conflict items.
Significant features undergo Ma et al. (2026) falsification to filter out
token-level artifacts.

**What this notebook does:**
1. Load activations and encode through SAE
2. Run differential activation tests (Mann-Whitney U, BH-FDR)
3. Volcano plot: log fold change vs -log10(q-value)
4. Ma et al. falsification of top features
5. Matched-pair subset analysis (difficulty-controlled)
6. Summary of genuine S1/S2-specific features

**Modes:**
- **Synthetic** (default): uses MockSAE on synthetic activations
- **Real**: load actual SAEs (Llama Scope / Gemma Scope) on real activations

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from s1s2.utils.io import (
    open_activations, get_residual, load_problem_metadata,
    list_models, model_metadata,
)
from s1s2.sae import (
    MockSAE, encode_batched,
    differential_activation, matched_pair_subset,
    plot_volcano, DifferentialResult,
    ma_et_al_falsification, falsify_candidates,
)
from s1s2.viz.theme import set_paper_theme, COLORS

set_paper_theme()

In [ ]:
# ---- Mode selection ----
USE_SYNTHETIC = True  # Set to False for real data

if USE_SYNTHETIC:
    from tests.conftest import build_synthetic_hdf5, SYNTH_MODEL_KEY
    DATA_PATH = REPO_ROOT / "data" / "activations" / "synthetic_nb03.h5"
    build_synthetic_hdf5(DATA_PATH)
    MODEL_KEY = SYNTH_MODEL_KEY
    LAYER = 2  # Planted signal layer
    POSITION = "P0"
    print(f"Built synthetic HDF5 at {DATA_PATH}")
    print(f"Using MockSAE (random encoder) -- features will be noisy.")
else:
    DATA_PATH = REPO_ROOT / "data" / "activations" / "main.h5"  # REQUIRES REAL DATA
    MODEL_KEY = "meta-llama_Llama-3.1-8B-Instruct"  # REQUIRES REAL DATA
    LAYER = 16  # Middle layer -- adjust based on probe results  # REQUIRES REAL DATA
    POSITION = "P0"
    print(f"Using real data at {DATA_PATH}")

In [ ]:
# --- Load activations ---
with open_activations(DATA_PATH) as f:
    meta = load_problem_metadata(f)
    mm = model_metadata(f, MODEL_KEY)
    X = get_residual(f, MODEL_KEY, LAYER, position=POSITION)

conflict = meta["conflict"]
hidden_dim = int(mm["hidden_dim"])

print(f"Model: {MODEL_KEY}")
print(f"Layer: {LAYER}, Position: {POSITION}")
print(f"Activations shape: {X.shape}")
print(f"Conflict items: {conflict.sum()}, Control items: {(~conflict).sum()}")

## Load or Create SAE

In [ ]:
if USE_SYNTHETIC:
    # MockSAE: random encoder for testing the pipeline
    n_features = 256  # Small for speed
    sae = MockSAE(input_dim=hidden_dim, n_features=n_features, seed=42)
    print(f"MockSAE: {hidden_dim} -> {n_features} features")
else:
    # REQUIRES REAL DATA
    # Load pre-trained SAE for this model/layer
    from s1s2.sae import load_sae_for_model, reconstruction_report
    sae = load_sae_for_model(MODEL_KEY, layer=LAYER)  # REQUIRES REAL DATA
    # Reconstruction fidelity gate (CLAUDE.md: always check before trusting)
    report = reconstruction_report(sae, X[:50])  # REQUIRES REAL DATA
    print(f"Reconstruction: MSE={report.mse:.4f}, "
          f"explained_var={report.explained_variance:.3f}")
    if report.explained_variance < 0.85:
        print("WARNING: low reconstruction fidelity. SAE may not fit this model.")

## Encode and Run Differential Activation

In [ ]:
# Encode activations through SAE
codes = encode_batched(sae, X.astype(np.float32), batch_size=128)
print(f"SAE codes shape: {codes.shape}")
print(f"Non-zero rate: {(codes > 0).mean():.3f}")
print(f"Mean activation (non-zero): {codes[codes > 0].mean():.3f}")

In [ ]:
# Differential activation: conflict vs control
result_all = differential_activation(
    codes=codes,
    conflict=conflict,
    fdr_q=0.05,
    subset="all",
)

n_sig = result_all.df["significant"].sum()
print(f"Significant features (q<0.05): {n_sig}/{len(result_all.df)}")
print(f"  S1-enriched (positive log_fc): "
      f"{(result_all.df['significant'] & (result_all.df['log_fc'] > 0)).sum()}")
print(f"  S2-enriched (negative log_fc): "
      f"{(result_all.df['significant'] & (result_all.df['log_fc'] < 0)).sum()}")

## Volcano Plot

In [ ]:
# Volcano plot
volcano_path = REPO_ROOT / "figures" / "sae_volcano_nb03.png"
volcano_path.parent.mkdir(parents=True, exist_ok=True)

fig = plot_volcano(
    result_all.df,
    title=f"{MODEL_KEY} layer {LAYER} ({POSITION}, all pairs)",
    out_path=volcano_path,
    fdr_q=0.05,
    annotate_top_k=5,
)
plt.show()
print(f"Saved to {volcano_path}")

## Matched-Pair Subset Analysis

Re-run differential activation on the matched-difficulty subset only. Per CLAUDE.md,
this is the result to quote when the all-pairs and matched results disagree.

In [ ]:
# Extract matched pair indices
matched_mask = matched_pair_subset(
    conflict=conflict,
    matched_pair_ids=meta["matched_pair_id"],
)

if matched_mask.sum() > 0:
    result_matched = differential_activation(
        codes=codes[matched_mask],
        conflict=conflict[matched_mask],
        fdr_q=0.05,
        subset="matched_pairs",
    )

    n_sig_matched = result_matched.df["significant"].sum()
    print(f"Matched subset: {matched_mask.sum()} items")
    print(f"Significant features (matched): {n_sig_matched}/{len(result_matched.df)}")

    # Compare all vs matched
    if n_sig > 0:
        sig_all = set(result_all.df[result_all.df["significant"]]["feature_id"])
        sig_matched = set(result_matched.df[result_matched.df["significant"]]["feature_id"])
        overlap = sig_all & sig_matched
        print(f"\nOverlap: {len(overlap)} features significant in both")
        print(f"  All-only: {len(sig_all - sig_matched)}")
        print(f"  Matched-only: {len(sig_matched - sig_all)}")
else:
    print("No matched pairs found (expected in some synthetic configs).")

## Ma et al. (2026) Falsification

For each significant feature: inject the top-3 activating tokens into 100 random
non-cognitive-bias texts. If the feature still activates, it is a token-level
artifact and should be discounted.

This step is NON-NEGOTIABLE per CLAUDE.md.

In [ ]:
if USE_SYNTHETIC:
    # In synthetic mode, demonstrate the falsification API with mock data
    print("Synthetic mode: demonstrating falsification API structure.")
    print("With real data, this cell runs the full Ma et al. pipeline.")
    print()

    # Show what the output looks like
    sig_features = result_all.df[result_all.df["significant"]]
    if len(sig_features) > 0:
        print(f"Top 5 significant features by |log_fc|:")
        top5 = sig_features.reindex(
            sig_features["log_fc"].abs().sort_values(ascending=False).index
        ).head(5)
        display(top5[["feature_id", "log_fc", "q_value", "effect_size",
                       "mean_S1", "mean_S2"]].reset_index(drop=True))
    else:
        print("No significant features found (expected with random MockSAE).")
else:
    # REQUIRES REAL DATA -- run full falsification
    sig_feature_ids = result_all.df[result_all.df["significant"]]["feature_id"].tolist()
    if sig_feature_ids:
        falsification = falsify_candidates(  # REQUIRES REAL DATA
            sae=sae,
            codes=codes,
            feature_ids=sig_feature_ids,
            n_random_texts=100,
        )
        n_falsified = sum(1 for r in falsification if r.is_falsified)
        print(f"Falsified (token-level artifacts): {n_falsified}/{len(sig_feature_ids)}")
        print(f"Surviving genuine features: {len(sig_feature_ids) - n_falsified}")

        # Annotate the main dataframe
        falsified_ids = {r.feature_id for r in falsification if r.is_falsified}
        result_all.df["is_falsified"] = result_all.df["feature_id"].isin(falsified_ids)

        # Re-plot volcano with falsification overlay
        fig = plot_volcano(
            result_all.df,
            title=f"{MODEL_KEY} layer {LAYER} ({POSITION}, with falsification)",
            out_path=REPO_ROOT / "figures" / "sae_volcano_falsified_nb03.png",
        )
        plt.show()
    else:
        print("No significant features to falsify.")

## Feature Activation Distributions

In [ ]:
# Show activation distributions for the top feature
sig_features = result_all.df[result_all.df["significant"]]

if len(sig_features) > 0:
    top_features = sig_features.reindex(
        sig_features["log_fc"].abs().sort_values(ascending=False).index
    ).head(4)

    fig, axes = plt.subplots(1, min(4, len(top_features)),
                             figsize=(4 * min(4, len(top_features)), 3),
                             squeeze=False)
    axes = axes[0]

    for idx, (_, row) in enumerate(top_features.iterrows()):
        fid = int(row["feature_id"])
        ax = axes[idx]
        ax.hist(codes[conflict, fid], bins=20, alpha=0.7,
                label="Conflict", color=COLORS["s1"], density=True)
        ax.hist(codes[~conflict, fid], bins=20, alpha=0.7,
                label="Control", color=COLORS["s2"], density=True)
        ax.set_title(f"Feature {fid}\nlog_fc={row['log_fc']:.2f}", fontsize=9)
        ax.set_xlabel("Activation")
        if idx == 0:
            ax.legend(fontsize=7)

    plt.suptitle("Top differential features: activation distributions", fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("No significant features to visualize.")
    print("(Expected with MockSAE on random noise. Real SAEs will show genuine features.)")

## Summary

**Key outputs:**

1. **Volcano plot**: the canonical summary. Features in the top-left/top-right corners are S2-/S1-enriched.
2. **Matched-pair analysis**: controls for difficulty confound. The matched result is definitive.
3. **Ma et al. falsification**: any feature that passes FDR but fails falsification is a token-level artifact, not a genuine processing-mode feature.

**Interpretation guide:**
- Genuine S1/S2 features: significant in both all-pairs AND matched-pairs, and NOT falsified
- Confounded features: significant only in all-pairs (difficulty confound)
- Token artifacts: significant but falsified (fire on top tokens regardless of context)

**Next steps:** Inspect top surviving features' activating tokens to build mechanistic hypotheses. Cross-reference with probing peak layers (notebook 02).